# EVOLVE — YOLO Evaluation Notebook

This notebook is dedicated to the **evaluation and qualitative analysis** of
the YOLO-based object detection model trained on the EVOLVE dataset.

The objectives are:
- quantitative evaluation (mAP, precision, recall)
- per-class performance analysis
- qualitative inspection of detections
- identification of common failure modes

All dataset design choices and annotation rules are documented in
`evolve_workbook.qmd`.

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from pathlib import Path
import random
import pandas as pd
from PIL import Image

## Load trained model

The model corresponds to the best weights obtained during training.

In [ ]:
model = YOLO("runs/evolve_yolo/weights/best.pt")

## Quantitative evaluation

We evaluate the model on the validation or test split using standard
object detection metrics.

In [ ]:
metrics = model.val(
    data="data/yolo/dataset.yaml"
)

metrics

## Per-class Average Precision (AP)

Beyond global mAP values, we inspect detection performance **per class**
to better understand model behavior under extreme visual conditions.

In [ ]:
names = metrics.names
ap50 = metrics.box.ap50
ap = metrics.box.ap

per_class_ap = {
    names[i]: {
        "AP@50": float(ap50[i]),
        "AP@50:95": float(ap[i])
    }
    for i in range(len(names))
}

per_class_ap

In [ ]:
df_ap = pd.DataFrame(per_class_ap).T
df_ap

### Interpretation of per-class performance

Several trends emerge from the per-class AP values:

- **Higher AP** for large, static objects such as `amp` and `drums`
- **Lower AP** for small or thin objects like `micro`
- **Coarse but semantically meaningful performance** for collective patterns
  such as `mosh_pit` and `hands_raised`

These differences are consistent with object scale variability,
illumination degradation, motion blur, and annotation granularity.

## Qualitative evaluation on test images

We visually inspect model predictions on a subset of test images
to identify typical success cases and failure modes.

In [ ]:
test_images = list(Path("data/yolo/images/test").glob("*.jpg"))
sample_images = random.sample(test_images, k=4)
sample_images

In [ ]:
results = model.predict(
    source=[str(p) for p in sample_images],
    conf=0.25,
    save=False
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, res in zip(axes.flatten(), results):
    img = Image.fromarray(res.plot()[:, :, ::-1])
    ax.imshow(img)
    ax.axis("off")

plt.tight_layout()

### Qualitative observations

The qualitative examples illustrate several recurring behaviors:

- Correct localization of large stage objects despite extreme lighting
- Missed detections for small or partially occluded microphones
- Coarse but relevant localization of crowd dynamics
- Occasional confusion between visually similar classes (`guitar` vs `micro`)

These observations confirm that **quantitative metrics alone are insufficient**
to fully characterize model behavior in extreme environments.

## Discussion: object scale and annotation granularity

Some target classes (e.g. `mosh_pit`, `hands_raised`) do not correspond
to well-defined physical objects.

As a result:
- bounding boxes are intentionally coarse,
- IoU-based metrics may underestimate semantic correctness,
- qualitative inspection is essential to complement quantitative evaluation.

This highlights the importance of adapting evaluation practices
when applying object detection models to non-standard visual domains.

## End of evaluation notebook

This notebook completes the experimental pipeline by combining
quantitative metrics with qualitative and semantic analysis.

The results are discussed in the final project report.